# 08 — Grounded Product QA

This notebook only wires together the production modules.

Core logic lives in:

- `src/rag/generation/`
- `src/rag/pipeline/qa.py`
- existing BM25 / FAISS / Hybrid retrievers

The LLM receives only retrieved real comments and must return
evidence comment IDs with every supported answer.

In [5]:
from pathlib import Path
import sys
import os
import json

from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.rag.config import load_config
from src.rag.preprocessing.processor import TextProcessor
from src.rag.embedding.factory import EmbeddingFactory
from src.rag.vector_store.faiss import FAISSVectorStore
from src.rag.retrieval.embedding import EmbeddingRetriever
from src.rag.retrieval.bm25 import BM25Retriever
from src.rag.retrieval.hybrid import HybridRetriever
from src.rag.generation import OpenAIJSONGenerator
from src.rag.pipeline import GroundedQAPipeline

load_dotenv()

True

In [7]:
RAG_CONFIG_PATH = (
    PROJECT_ROOT
    / "configs"
    / "rag.yaml"
)

QA_CONFIG_PATH = (
    PROJECT_ROOT
    / "configs"
    / "qa.yaml"
)

FAISS_PATH = (
    PROJECT_ROOT
    / "data"
    / "indexes"
    / "product_comments_embedding"
)

BM25_PATH = (
    PROJECT_ROOT
    / "data"
    / "indexes"
    / "product_comments_bm25_tantivy"
)

EVAL_PATH = (
    PROJECT_ROOT
    / "data"
    / "evaluation"
    / "retrieval_queries.json"
)

rag_config = load_config(
    RAG_CONFIG_PATH
)

qa_config = load_config(
    QA_CONFIG_PATH
)

## Load production retrievers

In [8]:
processor = TextProcessor()

embedding_model = (
    EmbeddingFactory.create(
        provider=rag_config[
            "embedding"
        ]["provider"],
        model_name=rag_config[
            "embedding"
        ]["model"],
    )
)

vector_store = FAISSVectorStore()
vector_store.load(
    FAISS_PATH
)

embedding_retriever = (
    EmbeddingRetriever(
        embedding_model=(
            embedding_model
        ),
        processor=processor,
        vector_store=(
            vector_store
        ),
    )
)

bm25_retriever = (
    BM25Retriever(
        processor=processor
    )
)
bm25_retriever.load(
    BM25_PATH
)

hybrid_config = (
    rag_config[
        "retrieval"
    ]["hybrid"]
)

hybrid_retriever = (
    HybridRetriever(
        bm25_retriever=(
            bm25_retriever
        ),
        embedding_retriever=(
            embedding_retriever
        ),
        bm25_weight=(
            hybrid_config[
                "bm25_weight"
            ]
        ),
        embedding_weight=(
            hybrid_config[
                "embedding_weight"
            ]
        ),
    )
)

Loading weights: 100%|████████████████████| 199/199 [00:00<00:00, 15979.07it/s]


## Create generator and QA pipeline

In [9]:
generator = OpenAIJSONGenerator(
    api_key=os.getenv(
        "METIS_API_KEY"
    ),
    base_url=os.getenv(
        "METIS_BASE_URL"
    ),
    model=qa_config[
        "generation"
    ]["model"],
    input_cost_per_million=(
        qa_config[
            "generation"
        ].get(
            "input_cost_per_million"
        )
    ),
    output_cost_per_million=(
        qa_config[
            "generation"
        ].get(
            "output_cost_per_million"
        )
    ),
)

qa_pipeline = GroundedQAPipeline(
    retriever=hybrid_retriever,
    generator=generator,
    top_k=qa_config[
        "qa"
    ]["top_k"],
    max_context_chars=(
        qa_config[
            "qa"
        ]["max_context_chars"]
    ),
    max_chars_per_comment=(
        qa_config[
            "qa"
        ][
            "max_chars_per_comment"
        ]
    ),
)

## Smoke test using one evaluation query

In [10]:
with open(
    EVAL_PATH,
    encoding="utf-8"
) as file:
    eval_samples = json.load(
        file
    )

sample = eval_samples[0]

print(
    "Product:",
    sample["product_title"]
)
print(
    "Query:",
    sample["query"]
)

Product: کرم ضد آفتاب آردن سان مدل Light Beige مقدار 50 گرم
Query: این ضدآفتاب برای پوست چرب باعث جوش می‌شود؟


In [11]:
result = qa_pipeline.answer(
    query=sample["query"],
    product_id=sample[
        "product_id"
    ],
)

print("ANSWER:")
print(result["answer"])

print()
print(
    "Evidence IDs:",
    result["evidence_ids"]
)

print(
    "Confidence:",
    result["confidence"]
)

print(
    "Grounding valid:",
    result["grounding_valid"]
)

print(
    "Validation errors:",
    result[
        "validation_errors"
    ]
)

ANSWER:
بر اساس نظرها، برای پوست چرب چند نفر گفته‌اند با این ضدآفتاب جوش نزده‌اند؛ اما تجربه همه افراد یکسان نیست و تضمینی برای پوست شما نمی‌شود.

Evidence IDs: [37147768, 53850976]
Confidence: medium
Grounding valid: True
Validation errors: []


In [12]:
display(
    result[
        "evidence_documents"
    ][
        [
            "id",
            "rate",
            "body",
            "score",
        ]
    ]
)

result["telemetry"]

,id,rate,body,score
0,37147768,5.0,برای پوست چرب خوبه جوش نمی زنه و بی رنگ هست رو...,0.792402
1,53850976,4.0,سبکه بوی خوبی داره پوستم چربه خوشبختانه باهاش ...,0.447860


{'retrieval_latency_ms': 630.6777949998832,
 'generation_latency_ms': 2473.070305999954,
 'total_latency_ms': 3111.0325559998273,
 'model': 'gpt-5.6-terra',
 'prompt_tokens': 601,
 'completion_tokens': 73,
 'total_tokens': 674,
 'estimated_cost_usd': None}

## Manual tests

Test several question types for the same product:

- positive experience
- repeated complaint
- ambiguous/conflicting evidence
- a question that is not supported by retrieved comments

The last case should return `insufficient_evidence=true`
rather than inventing an answer.

In [13]:
custom_result = qa_pipeline.answer(
    query="ایرادهای پرتکرار این محصول چیست؟",
    product_id=sample[
        "product_id"
    ],
)

print(
    custom_result["answer"]
)

display(
    custom_result[
        "evidence_documents"
    ][
        [
            "id",
            "rate",
            "body",
            "score",
        ]
    ]
)

custom_result[
    "telemetry"
]

ایرادهای پرتکرار بیشتر مربوط به تاریخ و تازگی کالا است: چند نفر از تاریخ تولید قدیمی، نزدیک‌بودن/گذشتن تاریخ انقضا یا درج‌نشدن تاریخ شکایت کرده‌اند. همچنین در یک تجربه، کرم روی صورت ماسیده و باعث جوش و لک شده است؛ بنابراین واکنش پوستی برای همه یکسان نیست.


,id,rate,body,score
0,38398473,2.0,تاریخ تولیدش سال ۹۹ بود، هنوز بازش نکردم ببینم...,0.430962
1,16407002,0.0,از دیجی کالا انتظار می‌رود درخصوص تاریخ تولید ...,0.356790
2,28598474,3.0,خواهش میکنم کرم ضد آفتاب خوب بفرستین این کرمی ...,0.327730
3,22330406,3.0,با هر بار استفاده از این کرم صورتم پر جوش میشد...,0.360475


{'retrieval_latency_ms': 323.58213499992416,
 'generation_latency_ms': 3512.876204999884,
 'total_latency_ms': 3839.852603000054,
 'model': 'gpt-5.6-terra',
 'prompt_tokens': 742,
 'completion_tokens': 120,
 'total_tokens': 862,
 'estimated_cost_usd': None}